# AI Resume Intelligence — Automated Resume Scoring

This notebook breaks down the algorithmic Scoring Engine used in the backend (`backend/app/utils/scoring.py`).
It takes the structured data extracted by the NLP pipeline and the Job Role predicted by the ML model, and computes a hiring recommendation based on specific Job Requirements.

**Scoring Weights:**
- **Skills (40%)**: Required Skills (30%) + Preferred Skills (10%)
- **Experience Match (25%)**: Calculates total years of experience against a minimum requirement.
- **Job Role Match (20%)**: Compares the AI-predicted job role with the target job role.
- **Education & Abilities (15%)**: Bonus points for having listed education and extra abilities.

In [2]:
!pip install re, datetime

ERROR: Invalid requirement: 're,': Expected semicolon (after name with no version specifier) or end
    re,
      ^


In [3]:
import re
from datetime import datetime
import json

### Step 1: Mock Data Setup
Since we don't have a database connection here, we will create some mock classes to simulate how a `Candidate` and their related data (Skills, Experiences, Education) are structured in the system.

In [4]:
class MockSkill:
    def __init__(self, name):
        self.name = name

class MockExperience:
    def __init__(self, title, start_date, end_date):
        self.title = title
        self.start_date = start_date
        self.end_date = end_date

class MockCandidate:
    def __init__(self):
        self.name = "Jane Doe"
        self.predicted_job_role = "Senior Software Engineer"  # From ML Model
        self.skills = [
            MockSkill("Python"), MockSkill("React"), MockSkill("SQL"), 
            MockSkill("AWS"), MockSkill("Docker")
        ]
        self.experiences = [
            MockExperience("Software Engineer", "01/2020", "06/2023"),
            MockExperience("Senior Software Engineer", "06/2023", "Present")
        ]
        self.educations = ["B.S. Computer Science"]
        self.abilities = ["Agile Methodology", "System Design"]

candidate = MockCandidate()

### Step 2: Experience Calculation
The system parses dates from resume text (like '01/2020' or 'Present') and computes the total duration of experience in years.

In [5]:
def parse_date(date_str: str) -> datetime:
    if not date_str or date_str.lower() in ['present', 'current']:
        return datetime.utcnow()
    
    # Extract years (e.g. 2020)
    match = re.search(r'\d{4}', date_str)
    if match:
        year = int(match.group())
        return datetime(year, 1, 1)
    return datetime.utcnow()

def calculate_total_experience(experiences) -> float:
    total_years = 0.0
    for exp in experiences:
        if not exp.start_date:
            continue
            
        start = parse_date(exp.start_date)
        end = parse_date(exp.end_date)
        
        duration_days = (end - start).days
        if duration_days > 0:
            total_years += duration_days / 365.25
            
    return round(total_years, 1)

total_exp = calculate_total_experience(candidate.experiences)
print(f"Total calculated experience: {total_exp} years")

Total calculated experience: 6.7 years


C:\Users\Gaurav Patel\AppData\Local\Temp\ipykernel_30428\1507054012.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow()


### Step 3: The Scoring Algorithm
We evaluate the candidate against a specific Job opening's requirements.

In [6]:
def score_candidate(candidate, target_role, req_skills, pref_skills, min_exp):
    score = 0.0
    breakdown = {}
    
    # Lowercase inputs for case-insensitive matching
    target_role_lower = target_role.lower()
    req_skills_lower = [s.lower() for s in req_skills]
    pref_skills_lower = [s.lower() for s in pref_skills]
    candidate_skills = [s.name.lower() for s in candidate.skills]
    
    # 1. Skills (40%)
    matched_req = [rs for rs in req_skills_lower if rs in candidate_skills]
    missing_req = [rs for rs in req_skills_lower if rs not in candidate_skills]
    
    req_score = (len(matched_req) / len(req_skills_lower)) * 30.0 if req_skills_lower else 30.0
    
    matched_pref = [ps for ps in pref_skills_lower if ps in candidate_skills]
    pref_score = (len(matched_pref) / len(pref_skills_lower)) * 10.0 if pref_skills_lower else 10.0
    
    skills_score = req_score + pref_score
    score += skills_score
    breakdown["skills_score"] = round(skills_score, 1)
    
    # 2. Experience Match (25%)
    exp_years = calculate_total_experience(candidate.experiences)
    exp_score = 25.0
    if min_exp and min_exp > 0:
        if exp_years >= min_exp:
            exp_score = 25.0
        else:
            exp_score = (exp_years / min_exp) * 25.0
            
    score += exp_score
    breakdown["experience_score"] = round(exp_score, 1)
    
    # 3. Job Role Match (20%)
    role_score = 0.0
    # Checking the AI Predicted Role
    if candidate.predicted_job_role and target_role_lower in candidate.predicted_job_role.lower():
        role_score = 20.0
    else:
        # Check Past Experience Titles as fallback
        past_titles = [e.title.lower() for e in candidate.experiences if e.title]
        if any(target_role_lower in t for t in past_titles):
            role_score = 15.0
            
    score += role_score
    breakdown["role_score"] = round(role_score, 1)
    
    # 4. Education/Abilities (15%)
    edu_score = 10.0 if candidate.educations else 0.0
    ab_score = 5.0 if candidate.abilities else 0.0
    
    edu_ab_score = edu_score + ab_score
    score += edu_ab_score
    breakdown["education_abilities_score"] = round(edu_ab_score, 1)
    
    # Finalizing Score
    breakdown["total_score"] = round(score, 1)
    breakdown["matched_required"] = matched_req
    breakdown["missing_required"] = missing_req
    
    # Generating Analysis & Recommendation
    strengths = []
    if skills_score >= 35: strengths.append("Exceptional technical skill match.")
    if exp_score >= 20: strengths.append("Meets or exceeds experience requirements.")
    if role_score >= 15: strengths.append("Proven background in the target role.")
        
    weak_areas = []
    if missing_req: weak_areas.append(f"Missing key required skills: {', '.join(missing_req)}")
    if exp_score < 15 and min_exp: weak_areas.append("Falls short of requested experience.")
        
    recommendation = "Consider"
    if score >= 80:
        recommendation = "Strong Hire"
    elif score < 60:
        recommendation = "Reject"
        
    breakdown["analysis"] = {
        "strengths": " ".join(strengths) if strengths else "No major strengths identified.",
        "weak_areas": " ".join(weak_areas) if weak_areas else "No significant weak areas identified.",
        "recommendation": recommendation
    }
    
    return breakdown

### Step 4: Run the Algorithm
Let's test our `Jane Doe` against a Job Description for a **Software Engineer** requiring Python, React, and Java.

In [ ]:
job_role = "Software Engineer"
required_skills = ["Python", "React", "Java"]
preferred_skills = ["AWS", "GCP"]
minimum_experience_years = 3

result = score_candidate(
    candidate=candidate,
    target_role=job_role,
    req_skills=required_skills,
    pref_skills=preferred_skills,
    min_exp=minimum_experience_years
)

print("Final Evaluation:")
print(json.dumps(result, indent=2))

Final Evaluation:
{
  "skills_score": 25.0,
  "experience_score": 25.0,
  "role_score": 20.0,
  "education_abilities_score": 15.0,
  "total_score": 85.0,
  "matched_required": [
    "python",
    "react"
  ],
  "missing_required": [
    "java"
  ],
  "analysis": {
    "strengths": "Meets or exceeds experience requirements. Proven background in the target role.",
    "weak_areas": "Missing key required skills: java",
    "recommendation": "Strong Hire"
  }
}


C:\Users\Gaurav Patel\AppData\Local\Temp\ipykernel_30428\1507054012.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow()


: 